# MAX5885 PL UART MMIO Control

AXI UART 16550 on right expansion header W9(RX)/W10(TX), 115200 8E1.

**Uses MMIO direct register access — no kernel driver needed.**
All serial parameters configurable at runtime via 16550 registers.

Deploy these files into the same Jupyter directory:
- base_add.bit and base_add.hwh
- pynqz1_diansai2_max5885.py

Each valid STM32 WCFG command configures MAX5885 and pulses LD1 for 500 ms.

In [ ]:
# Cell 1: load overlay, init 16550 via MMIO, define TX/RX helpers.
from time import sleep, time
import os
from pynq import Overlay, MMIO
from pynqz1_diansai2_max5885 import MAX5885Signal

BITFILE = 'base_add.bit'
# Addresses from base_add.hwh MEMORYMAP (verified 2026-07-16):
UART_BASE    = 0x43C00000   # uart_0 — Vivado auto-assigned, NOT 0x40002000
LED_BASE     = 0x40000000   # led_ctrl_0
MAX5885_BASE = 0x40001000   # max5885_ctrl_0

# ---- load overlay ----
overlay = Overlay(BITFILE)
print('Overlay:', BITFILE)
print('IP blocks:', list(overlay.ip_dict.keys()))

# ---- MMIO handles ----
uart = MMIO(UART_BASE, 0x10000)   # uart_0 range = 64KB
led  = MMIO(LED_BASE, 0x1000)
dac  = MAX5885Signal(MAX5885_BASE)

print('DAC version=0x%08X, sample_rate=%d' % (dac.status()['version'], dac.status()['sample_rate']))

# ============================================================
#  AXI UART 16550 v2.0 register map (byte offsets, 32-bit access)
# ============================================================
RBR_THR_DLL = 0x00   # RBR (read, DLAB=0) / THR (write, DLAB=0) / DLL (r/w, DLAB=1)
IER_DLM     = 0x04   # IER (r/w, DLAB=0) / DLM (r/w, DLAB=1)
FCR_IIR     = 0x08   # IIR (read) / FCR (write)
LCR         = 0x0C   # Line Control Register
MCR         = 0x10   # Modem Control Register
LSR         = 0x14   # Line Status Register
MSR         = 0x18   # Modem Status Register
SCR         = 0x1C   # Scratch Register

# LSR bits
LSR_RX_READY = 0x01
LSR_TX_EMPTY = 0x20
LSR_RX_ERR   = 0x80  # parity/frame/break error summary

# LCR values
LCR_8E1  = 0x1B   # 8 data + even parity + 1 stop
LCR_8N1  = 0x03   # 8 data + no parity + 1 stop
LCR_8O1  = 0x0B   # 8 data + odd parity + 1 stop
LCR_DLAB = 0x80   # Divisor Latch Access Bit

# ============================================================
#  Init 16550: 115200 8E1, FIFO on
# ============================================================
AXI_CLK_HZ = 125_000_000
BAUD       = 115200
divisor    = round(AXI_CLK_HZ / (16 * BAUD))

# 1. Set DLAB, write divisor
uart.write(LCR, LCR_DLAB)
uart.write(RBR_THR_DLL, divisor & 0xFF)
uart.write(IER_DLM, (divisor >> 8) & 0xFF)
# 2. Set 8E1, clear DLAB
uart.write(LCR, LCR_8E1)
# 3. Enable + clear TX/RX FIFO
uart.write(FCR_IIR, 0x07)
# 4. Disable all interrupts (polling mode)
uart.write(IER_DLM, 0x00)

actual_baud = AXI_CLK_HZ / (16 * divisor)
print('16550 init: divisor=%d -> %d baud (error %.2f%%)' % (
    divisor, actual_baud, (actual_baud - BAUD) / BAUD * 100))

# ============================================================
#  UART TX/RX helpers
# ============================================================
def uart_tx_byte(b):
    while not (uart.read(LSR) & LSR_TX_EMPTY):
        pass
    uart.write(RBR_THR_DLL, b)

def uart_rx_byte(timeout_ms=200):
    deadline = time() + timeout_ms / 1000.0
    while time() < deadline:
        lsr = uart.read(LSR)
        if lsr & LSR_RX_READY:
            b = uart.read(RBR_THR_DLL) & 0xFF
            if lsr & LSR_RX_ERR:
                print('  UART RX error: LSR=0x%02X' % (lsr & 0x7F))
            return b
    return None

def uart_send(line):
    line = str(line).strip('\r\n')
    wire = (line + '\r\n').encode('ascii')
    for b in wire:
        uart_tx_byte(b)
    print('TX:', line)

def uart_read_line(timeout_s=1.0):
    deadline = time() + timeout_s
    raw = bytearray()
    while time() < deadline:
        ch = uart_rx_byte(timeout_ms=100)
        if ch is not None:
            if ch == 0x0A:   # \n
                line = raw.rstrip(b'\r').decode('ascii', errors='replace')
                print('RX:', line)
                return line
            if len(raw) < 255:
                raw.append(ch)
            else:
                raw.clear()  # overflow, discard
    return None

def uart_drain(duration_s=0.15):
    lines = []
    deadline = time() + duration_s
    while time() < deadline:
        line = uart_read_line(timeout_s=min(0.05, deadline - time()))
        if line is not None:
            lines.append(line)
    return lines

# ============================================================
#  LED pulse helper (LD1 = bit1 of LED_VALUE)
# ============================================================
LED_CTRL   = 0x00
LED_VALUE  = 0x08
LED1_MASK  = 0x2

class LedPulse:
    def __init__(self):
        self.restore = None
        self.deadline = 0.0
    def pulse_ld1(self, duration_s=0.5):
        now = time()
        if self.restore is None:
            self.restore = led.read(LED_VALUE)
        led.write(LED_CTRL, 0)
        led.write(LED_VALUE, self.restore | LED1_MASK)
        self.deadline = now + duration_s
    def update(self):
        if self.restore is not None and time() >= self.deadline:
            led.write(LED_CTRL, 0)
            led.write(LED_VALUE, self.restore)
            self.restore = None

led_pulse = LedPulse()
print('MMIO UART ready: W9(RX)/W10(TX), 115200 8E1')

In [ ]:
# Cell 2: WCFG parser + MAX5885 apply.
# STM32 format (14 fields):
# WCFG,carrier,mod,square,mode,sd_vrms,am_depth,sm_delay,sm_phase,
#       sm_atten,dac_a_gain,dac_b_gain,out_a,out_b
VALID_OUTPUTS = {'SD', 'SM', 'SOUT', 'DC', 'SQUARE', 'MOD_SINE'}

def parse_wcfg(line):
    fields = [item.strip() for item in line.split(',')]
    if not fields or fields[0].upper() != 'WCFG':
        raise ValueError('not a WCFG line')
    if len(fields) != 14:
        raise ValueError('expected 14 fields, got %d' % len(fields))
    cfg = {
        'carrier_hz':       int(fields[1]),
        'mod_hz':           int(fields[2]),
        'square_hz':        int(fields[3]),
        'mode':             fields[4].upper(),
        'sd_vrms':          float(fields[5]),
        'am_depth_percent': float(fields[6]),
        'sm_delay_ns':      float(fields[7]),
        'sm_phase_deg':     float(fields[8]),
        'sm_atten_db':      float(fields[9]),
        'dac_a_gain':       float(fields[10]),
        'dac_b_gain':       float(fields[11]),
        'out_a':            fields[12].upper(),
        'out_b':            fields[13].upper(),
        'sd_phase_deg':     0.0,     # not in STM32 format, keep default
    }
    if cfg['mode'] not in ('CW', 'AM'):
        raise ValueError('mode must be CW or AM, got: ' + cfg['mode'])
    if cfg['out_a'] not in VALID_OUTPUTS or cfg['out_b'] not in VALID_OUTPUTS:
        raise ValueError('output must be one of: ' + ','.join(sorted(VALID_OUTPUTS)))
    return cfg

def apply_wcfg(line):
    try:
        cfg = parse_wcfg(line)
        actual = dac.configure_wireless(**cfg)
        led_pulse.pulse_ld1(0.5)
        uart_send('STATUS,PYNQ APPLIED %dMHz %s' % (round(cfg['carrier_hz']/1e6), cfg['mode']))
        uart_send('LOG,A=%s B=%s SD=%dmV' % (cfg['out_a'], cfg['out_b'], round(cfg['sd_vrms']*1000)))
        print('Applied: carrier=%.3fMHz mode=%s' % (actual['actual_carrier_hz']/1e6, cfg['mode']))
        return True
    except Exception as exc:
        uart_send('ERR,%s' % str(exc)[:80])
        print('ERR:', str(exc))
        return False

print('WCFG parser ready. Valid outputs:', ', '.join(sorted(VALID_OUTPUTS)))

In [ ]:
# Cell 3: link test. Sends STATUS/TEST, waits 10s for STM32 WCFG.
uart_drain()
uart_send('STATUS,PYNQ MMIO 16550 READY')
uart_send('TEST,UART TX OK 8E1')
uart_send('LOG,PL UART on W9(RX)/W10(TX)')

print('Waiting 10s for STM32 WCFG response...')
got_wcfg = False
deadline = time() + 10.0
while time() < deadline:
    led_pulse.update()
    line = uart_read_line(timeout_s=0.25)
    if line is None:
        continue
    if line.upper().startswith('WCFG,'):
        uart_send('TEST,UART RX OK 8E1')
        apply_wcfg(line)
        got_wcfg = True
        break

if not got_wcfg:
    print('No WCFG received. Check:')
    print('  STM32 TX -> W9 (FPGA RX)')
    print('  STM32 RX <- W10 (FPGA TX)')
    print('  GND -- GND')
    print('  STM32 firmware: 115200 8E1')
else:
    print('Link test PASSED. Run Cell 4 for continuous listening.')

In [ ]:
# Cell 4: continuous listener. Interrupt (Ctrl+C or Jupyter stop) to end.
print('Listening on PL UART (W9/W10, MMIO 16550). Each WCFG applies MAX5885 + pulses LD1.')
print('Interrupt this cell in Jupyter to stop.')
try:
    while True:
        led_pulse.update()
        line = uart_read_line(timeout_s=0.3)
        if line is None:
            continue
        if line.upper().startswith('WCFG,'):
            apply_wcfg(line)
except KeyboardInterrupt:
    print('Listener stopped.')

In [ ]:
# Cell 5 (optional): runtime baud/parity change example.
# Change the 16550 LCR and divisor on the fly without rebuilding bitstream.

def uart_reconfig(baud=115200, parity='E'):
    """Reconfigure 16550 baud rate and parity at runtime."""
    lcr_map = {
        'N': LCR_8N1,
        'E': LCR_8E1,
        'O': LCR_8O1,
    }
    if parity.upper() not in lcr_map:
        raise ValueError('parity must be N, E, or O')
    new_lcr = lcr_map[parity.upper()]
    new_div = round(AXI_CLK_HZ / (16 * int(baud)))
    
    uart.write(LCR, LCR_DLAB)
    uart.write(RBR_THR_DLL, new_div & 0xFF)
    uart.write(IER_DLM, (new_div >> 8) & 0xFF)
    uart.write(LCR, new_lcr)
    uart.write(FCR_IIR, 0x07)
    actual = AXI_CLK_HZ / (16 * new_div)
    print('Reconfigured: %d baud (actual %.0f), 8%s1' % (baud, actual, parity.upper()))

# Example: switch to 9600 8N1 then back
# uart_reconfig(9600, 'N')
# ... do something at 9600 ...
# uart_reconfig(115200, 'E')   # back to normal
print('uart_reconfig(baud=..., parity=...) available. Default: 115200 8E1.')